<a href="https://colab.research.google.com/github/zamoralinokevin-eng/Maestria-en-IA/blob/main/Actividad_Ejercicio2_OpenMeteo_Sesion8_271753.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad — Comparar el clima entre varias ciudades con Open-Meteo (Sesión 8)

En el Bloque 2 consultaste el pronóstico por hora de Ciudad Juárez con Open-Meteo. En esta actividad usas
la misma API, pero de otra forma: en vez de un pronóstico por hora de una sola ciudad, se consulta el
**clima actual** de **varias ciudades distintas** y se construye una tabla comparativa.

**Endpoint:** `https://api.open-meteo.com/v1/forecast`
**Parámetro clave:** `current_weather=true` — regresa un solo valor actual (no una lista por hora) dentro
de `datos["current_weather"]`, con `temperature`, `windspeed`, `weathercode` y `time`.

**Instrucciones:**

1. Define una lista `ciudades` con al menos tres ciudades (nombre, latitud, longitud), distintas a
   Ciudad Juárez.
2. Para cada ciudad, consulta el endpoint con `current_weather=true`, usando `timeout` y manejo de
   errores como en el Bloque 2.
3. De cada respuesta, extrae la temperatura y la velocidad del viento actuales.
4. Estructura la información como una lista de diccionarios llamada `info_clima`.
5. Convierte `info_clima` en un DataFrame llamado `df_clima_ciudades`.
6. Verifica el resultado.

**Paso 1 — Define las ciudades a consultar.**

In [16]:
ciudades = [
    {"nombre": "Chihuahua", "latitud": 28.63, "longitud": -106.09},
    {"nombre": "Monterrey", "latitud": 25.69, "longitud": -100.32},
    {"nombre": "Guadalajara", "latitud": 20.66, "longitud": -103.35},
    {"nombre": "Cancun", "latitud": 21.17, "longitud": -86.84},
]

**Pasos 2 a 4 — Consultar cada ciudad y estructurar la información.**

In [17]:
import requests

#se crea lista para extraer la temperatura y velocidad del viento
info_clima = []

for ciudad in ciudades:
    #se indexa los valores de cada ciudad, se llaman las llaves en c/iteracion
    endpoint = "https://api.open-meteo.com/v1/forecast"
    parametros = {
        "latitude": ciudad["latitud"],
        "longitude": ciudad["longitud"],
        "current_weather": True,
    }

    #envía la solicitud con timeout, revisa status_code con raise_for_status()
    try:
        respuesta = requests.get(endpoint, params=parametros, timeout=5)
        respuesta.raise_for_status()
        datos = respuesta.json()
        clima_actual = datos["current_weather"]

        #extrae temperature y windspeed de clima_actual
        info_clima.append({
            "ciudad": ciudad["nombre"],
            "temperatura": clima_actual["temperature"],
            "viento": clima_actual["windspeed"],
        })

    except requests.RequestException as error:
        print(f"No fue posible consultar {ciudad['nombre']}:", error)

info_clima

[{'ciudad': 'Chihuahua', 'temperatura': 24.4, 'viento': 3.3},
 {'ciudad': 'Monterrey', 'temperatura': 27.6, 'viento': 15.8},
 {'ciudad': 'Guadalajara', 'temperatura': 18.2, 'viento': 1.4},
 {'ciudad': 'Cancun', 'temperatura': 26.3, 'viento': 5.5}]

**Paso 5 — Construir el DataFrame.**

In [18]:
#convierte info_clima en un DataFrame llamado df_clima_ciudades
import pandas as pd

df_clima_ciudades = pd.DataFrame(info_clima)
df_clima_ciudades

,ciudad,temperatura,viento
0,Chihuahua,24.4,3.3
1,Monterrey,27.6,15.8
2,Guadalajara,18.2,1.4
3,Cancun,26.3,5.5


**Paso 6 — Verificar el resultado.**

In [19]:
#imprime shape, columns y valores faltantes de df_clima_ciudades
print(f"El dataframe tiene una forma de {df_clima_ciudades.shape}\n")
print(f"Las columnas son {df_clima_ciudades.columns.tolist()}\n")
print(f"Los valores faltantes/nulos son:\n{df_clima_ciudades.isnull().sum()}\n")

El dataframe tiene una forma de (4, 3)

Las columnas son ['ciudad', 'temperatura', 'viento']

Los valores faltantes/nulos son:
ciudad         0
temperatura    0
viento         0
dtype: int64

